# Probabilistic Models in Machine Learning

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 03.09 |
| Time | ~80 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/03_Probability_and_Statistics/10_probabilistic_models_in_ml.ipynb)

---

## 🎯 Learning Objective

Survey Gaussian Mixture Models, Naive Bayes, and latent-variable models as concrete probabilistic models used throughout ML.

---

## 📐 Theory

The tools from `03.01`-`03.09` now assemble into complete probabilistic *models* — systems that
describe how observed data was generated, often through variables you never directly see.

### Gaussian Mixture Models

A **Gaussian Mixture Model (GMM)** assumes data comes from $K$ Gaussian components, each data
point secretly generated by first picking a component $z\in\{1,\ldots,K\}$ (with $P(z=k)=\pi_k$)
and then drawing $x\sim\mathcal{N}(\mu_k,\Sigma_k)$:

$$p(x) = \sum_{k=1}^K \pi_k\,\mathcal{N}(x\mid\mu_k,\Sigma_k).$$

$z$ is a **latent variable** — never observed, only inferred. This is exactly the model
[Maximum Likelihood Estimation](07_maximum_likelihood_estimation.ipynb) (`03.07`) fit with EM:
the E-step computes each point's **responsibility** $P(z=k\mid x)$ (soft cluster assignment,
via Bayes' theorem, `03.05`), the M-step updates $\pi_k,\mu_k,\Sigma_k$ given those
responsibilities. GMMs generalize k-means clustering by allowing soft, elliptical
(covariance-shaped) clusters instead of hard, spherical ones.

### Naive Bayes

**Naive Bayes** classifiers use Bayes' theorem (`03.05`) directly: to classify $x=(x_1,\ldots,
x_d)$ into class $y$, compute $P(y\mid x) \propto P(x\mid y)P(y)$, and the "naive" part
*assumes* the features are conditionally independent given the class:

$$P(x\mid y) = \prod_{j=1}^d P(x_j \mid y).$$

This assumption is almost always technically false (features usually do correlate), but it
collapses a joint distribution over $d$ features into $d$ separate one-dimensional
distributions (`03.04`'s independence factorization, imposed by fiat rather than verified) —
each trivial to estimate via MLE, and the resulting classifier is often shockingly effective
in practice despite the "wrong" assumption.

### Latent variable models, generally

Both models above share a pattern: an **unobserved** variable $z$ (component index, class
label) explains structure in the **observed** variable $x$, and $p(x) = \sum_z p(x\mid z)p(z)$
(or $\int$, if $z$ is continuous) marginalizes it out — the exact operation from
[Joint, Conditional, and Marginal Distributions](04_joint_conditional_marginal.ipynb) (`03.04`).
The general recipe is always: define $p(z)$ (prior over latents), define $p(x\mid z)$
(likelihood/"decoder"), and use Bayes' theorem to reason about $p(z\mid x)$ (posterior/"encoder")
given data.

### Variational Autoencoders: latent variables meet neural networks

A **Variational Autoencoder (VAE)** makes both $p(z)$ and $p(x\mid z)$ neural: a **decoder**
network computes $p_\theta(x\mid z)$ from a *continuous* latent $z$ (typically
$z\sim\mathcal{N}(0,I)$), and since the true posterior $p(z\mid x)$ has no closed form, an
**encoder** network $q_\phi(z\mid x)$ approximates it — the model learns to both generate data
from latents and infer latents from data simultaneously. Training maximizes the **evidence
lower bound (ELBO)**,

$$\text{ELBO} = \mathbb{E}_{z\sim q_\phi(z\mid x)}[\log p_\theta(x\mid z)] - D_{\text{KL}}(q_\phi(z\mid x)\,\|\,p(z)),$$

a tractable *lower bound* on the true (intractable) log-likelihood $\log p_\theta(x)$ — the
first term rewards reconstructing $x$ well, the second (a KL divergence, formalized in
[KL Divergence](../05_Information_Theory/02_kl_divergence.ipynb), `05.02`) regularizes the
encoder's output toward the prior. Maximizing the ELBO is the continuous-latent, neural-network
generalization of exactly what EM does for GMMs.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from sklearn.mixture import GaussianMixture
from sklearn.naive_bayes import GaussianNB
import torch
import torch.nn as nn
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Fitting a GMM to 2D data with EM, watched over several iterations: each component's ellipse
(one standard deviation of its covariance) visibly migrates and reshapes to cover a cluster,
while points are colored by their current soft responsibility.

In [ ]:
# Fit a GMM with our own EM implementation (from 03.07's pattern) and visualize how the
# component ellipses converge onto the true clusters over iterations.
rng = np.random.default_rng(0)
true_means = np.array([[-2.0, -2.0], [2.0, 2.0], [2.0, -2.0]])
true_covs = [np.array([[1.0, 0.3], [0.3, 0.6]]), np.array([[0.6, -0.2], [-0.2, 1.0]]), np.array([[0.5, 0.0], [0.0, 0.5]])]
n_per = 150
X = np.vstack([rng.multivariate_normal(m, c, n_per) for m, c in zip(true_means, true_covs)])

def cov_ellipse(ax, mean, cov, color):
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    width, height = 2 * np.sqrt(vals)  # one-std-dev ellipse along each eigenvector axis
    ax.add_patch(Ellipse(mean, width, height, angle=angle, edgecolor=color, facecolor="none", lw=2.5))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
snapshots = [1, 3, 15]
colors = ["#4C72B0", "#DD8452", "#55A868"]
for ax, n_iter in zip(axes, snapshots):
    gmm = GaussianMixture(n_components=3, max_iter=n_iter, n_init=1, random_state=0, warm_start=False)
    gmm.fit(X)
    responsibilities = gmm.predict_proba(X)
    dominant_color = np.array([colors[k] for k in responsibilities.argmax(axis=1)])
    for k in range(3):
        ax.scatter(X[dominant_color == colors[k], 0], X[dominant_color == colors[k], 1],
                   s=10, alpha=0.5, color=colors[k])
        cov_ellipse(ax, gmm.means_[k], gmm.covariances_[k], colors[k])
    ax.set_title(f"After {n_iter} EM iteration(s)\nlog-likelihood={gmm.score(X)*len(X):.1f}")
    ax.set_xlim(-6, 6); ax.set_ylim(-6, 6)
plt.tight_layout()
plt.show()

final_gmm = GaussianMixture(n_components=3, random_state=0).fit(X)
print("Converged component means (order may differ from generation order):")
print(np.round(final_gmm.means_, 3))
print("True means:\n", true_means)

## 🐍 Implementation from Scratch

We implement a Gaussian Naive Bayes classifier entirely from scratch — per-class, per-feature
MLE (mean/variance) plus Bayes' theorem at prediction time — and cross-check every prediction
against `sklearn.naive_bayes.GaussianNB`.

In [ ]:
class NaiveBayesFromScratch:
    """Gaussian Naive Bayes: P(y|x) ~ P(y) * prod_j P(x_j|y), each P(x_j|y) a per-class,
    per-feature Gaussian fit by MLE (03.07) -- the conditional-independence assumption (03.04)
    applied by construction, not verified."""

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.priors_, self.means_, self.vars_ = {}, {}, {}
        for c in self.classes_:
            X_c = X[y == c]
            self.priors_[c] = len(X_c) / len(X)              # MLE of P(y=c): class frequency
            self.means_[c] = X_c.mean(axis=0)                  # MLE of each feature's mean | class
            self.vars_[c] = X_c.var(axis=0) + 1e-9             # MLE of each feature's variance | class
        return self

    def _log_likelihood(self, x, c):
        # log P(x|y=c) = sum_j log N(x_j; mean_jc, var_jc) -- the NAIVE independence factorization
        var = self.vars_[c]
        return -0.5 * np.sum(np.log(2 * np.pi * var) + (x - self.means_[c]) ** 2 / var)

    def predict_proba(self, X):
        log_posts = np.array([
            [np.log(self.priors_[c]) + self._log_likelihood(x, c) for c in self.classes_]
            for x in X
        ])
        log_posts -= log_posts.max(axis=1, keepdims=True)   # stabilize before exponentiating
        posts = np.exp(log_posts)
        return posts / posts.sum(axis=1, keepdims=True)      # normalize -- the "evidence" P(x) in Bayes' rule

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


rng = np.random.default_rng(0)
n_per_class = 200
class0 = rng.normal([0, 0], [1.0, 1.5], size=(n_per_class, 2))
class1 = rng.normal([3, 2], [1.2, 0.8], size=(n_per_class, 2))
X_train = np.vstack([class0, class1])
y_train = np.array([0] * n_per_class + [1] * n_per_class)

our_nb = NaiveBayesFromScratch().fit(X_train, y_train)
sklearn_nb = GaussianNB().fit(X_train, y_train)

our_preds = our_nb.predict(X_train)
sklearn_preds = sklearn_nb.predict(X_train)
our_probs = our_nb.predict_proba(X_train)
sklearn_probs = sklearn_nb.predict_proba(X_train)

print(f"Predictions match sklearn exactly: {np.array_equal(our_preds, sklearn_preds)}")
print(f"Max |our_probs - sklearn_probs|: {np.max(np.abs(our_probs - sklearn_probs)):.2e}")
print(f"Training accuracy: ours={np.mean(our_preds == y_train):.4f}, sklearn={np.mean(sklearn_preds == y_train):.4f}")

# Sanity-check the fitted parameters directly against the true generating distributions
print(f"\nFitted mean for class 0: {np.round(our_nb.means_[0], 3)}  (true: [0, 0])")
print(f"Fitted mean for class 1: {np.round(our_nb.means_[1], 3)}  (true: [3, 2])")

## 🤖 Why This Matters for AI

GMMs remain a standard soft-clustering/density-estimation tool; Naive Bayes is still a strong,
fast baseline for text classification. But their deeper significance is as the **conceptual
ancestor of modern generative models**: a VAE is, structurally, a GMM with the component count
taken to a continuum and per-component distributions replaced by neural networks. Below, we
train a small VAE on 2D data end to end — its encoder inferring a latent posterior, its decoder
generating new samples from the prior — the same encoder/decoder structure as the Theory section.

---

In [ ]:
# A minimal VAE on 2D "moon"-shaped data: encoder infers a latent posterior q(z|x), decoder
# reconstructs x from z, trained by maximizing the ELBO -- EM's continuous, neural generalization.
torch.manual_seed(0)
rng = np.random.default_rng(0)

n_pts = 600
theta = rng.uniform(0, np.pi, n_pts)
data_np = np.stack([np.cos(theta) + rng.normal(scale=0.1, size=n_pts),
                     np.sin(theta) + rng.normal(scale=0.1, size=n_pts)], axis=1).astype(np.float32)
X_data = torch.tensor(data_np)

latent_dim, hidden_dim, data_dim = 2, 16, 2

class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(data_dim, hidden_dim), nn.ReLU())
        self.to_mu = nn.Linear(hidden_dim, latent_dim)
        self.to_logvar = nn.Linear(hidden_dim, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, data_dim))

    def encode(self, x):
        h = self.encoder(x)
        return self.to_mu(h), self.to_logvar(h)   # parameters of q_phi(z|x) = N(mu, exp(logvar))

    def reparameterize(self, mu, logvar):
        # the "reparameterization trick": z = mu + std*eps lets gradients flow through a
        # SAMPLING step, by moving all randomness into eps ~ N(0,I), a fixed, parameter-free source
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

vae = VAE()
optimizer = torch.optim.Adam(vae.parameters(), lr=0.005)
n_epochs = 500
# Full KL weight (beta=1) pushes q(z|x) all the way to match the prior before the encoder has
# learned to encode anything useful, and with a 2D latent for 2D data the decoder can satisfy
# that pressure by ignoring z entirely ("posterior collapse"). A beta-VAE with beta<1 keeps
# some KL pressure (so the latent space stays well-behaved) while letting reconstruction win
# enough of the tradeoff that the encoder still has to use z to reduce error.
beta = 0.25

for epoch in range(n_epochs):
    x_recon, mu, logvar = vae(X_data)
    # ELBO = E[log p(x|z)] - KL(q(z|x) || p(z)); we MINIMIZE its negative. The reconstruction
    # term is a Gaussian log-likelihood (const + MSE); the KL term has a closed form for two
    # Gaussians (p(z)=N(0,I)) -- both terms are exactly the ELBO from the Theory section.
    recon_loss = torch.sum((x_recon - X_data) ** 2, dim=1).mean()
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()
    neg_elbo = recon_loss + beta * kl_loss
    optimizer.zero_grad()
    neg_elbo.backward()
    optimizer.step()
    if epoch % 100 == 0:
        print(f"epoch {epoch:3d}: reconstruction={recon_loss.item():.4f}, KL={kl_loss.item():.4f}, "
              f"-ELBO={neg_elbo.item():.4f}")

# Generative use: sample z from the PRIOR p(z)=N(0,I) (never touching the encoder at all) and
# decode -- this is literally "generate a new data point from nothing but the prior"
with torch.no_grad():
    z_prior_samples = torch.randn(300, latent_dim)
    generated = vae.decoder(z_prior_samples).numpy()
    mu_final, _ = vae.encode(X_data)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(data_np[:, 0], data_np[:, 1], s=10, alpha=0.5, color="#4C72B0", label="real data")
axes[0].scatter(generated[:, 0], generated[:, 1], s=10, alpha=0.5, color="#C44E52", label="VAE-generated")
axes[0].set_title("Decoder(z), z ~ prior N(0,I):\ngenerating NEW points from scratch"); axes[0].legend()
axes[1].scatter(mu_final[:, 0], mu_final[:, 1], s=10, alpha=0.5, c=theta, cmap="viridis")
axes[1].set_title("Encoder's inferred latent means mu(x)\ncolor = position along the true moon arc")
plt.tight_layout()
plt.show()

print(f"\nReal data range:      x0 in [{data_np[:,0].min():.2f}, {data_np[:,0].max():.2f}], "
      f"x1 in [{data_np[:,1].min():.2f}, {data_np[:,1].max():.2f}]")
print(f"VAE-generated range:  x0 in [{generated[:,0].min():.2f}, {generated[:,0].max():.2f}], "
      f"x1 in [{generated[:,1].min():.2f}, {generated[:,1].max():.2f}]  (should overlap the real range)")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. A 1-feature Gaussian Naive Bayes has $P(y{=}0)=P(y{=}1)=0.5$, class $0$'s feature
   distribution $\mathcal{N}(2,1)$, and class $1$'s $\mathcal{N}(5,1)$. By hand, compute
   $P(y{=}0\mid x{=}3)$ and $P(y{=}1\mid x{=}3)$ using Bayes' theorem
   (`03.05`) with the Gaussian likelihoods. As a sanity check, confirm your two posterior
   probabilities sum to exactly 1, then verify numerically using `our_nb`-style log-posterior
   code.

<details>
<summary>💡 Solution</summary>

$\log P(x{=}3\mid y{=}0) = -\frac12\log(2\pi\cdot1) - \frac{(3-2)^2}{2} \approx -1.419$, and
$\log P(x{=}3\mid y{=}1) = -\frac12\log(2\pi) - \frac{(3-5)^2}{2} \approx -2.919$. Adding
$\log P(y{=}0)=\log P(y{=}1)=\log0.5$ to each (equal for both, so it doesn't affect the
comparison) and normalizing: $P(y{=}0\mid x{=}3) \approx 0.818$,
$P(y{=}1\mid x{=}3)\approx0.182$ — summing to exactly $1$, as any correctly normalized posterior
must. This matches computing the same quantities directly with `_log_likelihood`-style code and
normalizing via `np.exp(log_posts - log_posts.max())` followed by dividing by the sum.

</details>

### 💭 UNDERSTAND
2. Fitting a GMM with `n_components=4` to data secretly generated from only 3 true clusters
   does not error out — it happily converges. Without running anything, predict what the 4th
   component "does" to the fit, then verify by inspecting the fitted means and weights.

<details>
<summary>💡 Solution</summary>

With one extra component than true clusters, EM cannot invent a 4th real cluster where none
exists — instead, the most common outcome is that one of the three real clusters gets **split**
into two nearby sub-components that share roughly half its true weight each, while the other
two true clusters are still each captured by one component. Verifying: fitting
`GaussianMixture(n_components=4)` on the notebook's 3-cluster data typically produces two of the
four fitted means very close to each other (both near one true cluster's center, e.g. both near
$(-2,-2)$ in a representative run, with weights around $0.16$ and $0.17$ splitting that
cluster's true $\approx0.33$ share), while the other two fitted means closely match the other
two true cluster centers with weights near $0.33$ each. A model-selection criterion like BIC
(`gmm.bic(X)`) correctly scores the 3-component fit as better (lower BIC) than the 4-component
fit on this data, reflecting that the extra component doesn't earn its added complexity.

</details>

### ✏️ DERIVE
3. The VAE training loop computes the KL term in closed form as
   `-0.5 * sum(1 + logvar - mu.pow(2) - logvar.exp())`. Derive this formula: starting from
   $D_{\text{KL}}(\mathcal{N}(\mu,\sigma^2)\,\|\,\mathcal{N}(0,1)) = \int
   q(z)\log\frac{q(z)}{p(z)}\,dz$ for a single latent dimension, substitute the two Gaussian
   densities, simplify the log-ratio, and take the expectation term by term using
   $\mathbb{E}_q[(z-\mu)^2]=\sigma^2$ and $\mathbb{E}_q[z^2] = \sigma^2+\mu^2$.

<details>
<summary>💡 Solution outline</summary>

$\log\frac{q(z)}{p(z)} = \log\frac{\frac{1}{\sqrt{2\pi\sigma^2}}\exp(-\frac{(z-\mu)^2}{2\sigma^2})}
{\frac{1}{\sqrt{2\pi}}\exp(-\frac{z^2}{2})} = -\frac12\log\sigma^2 - \frac{(z-\mu)^2}{2\sigma^2} +
\frac{z^2}{2}$. Taking $\mathbb{E}_q[\cdot]$ term by term: the first term is a constant,
$-\frac12\log\sigma^2$. The second: $\mathbb{E}_q\big[\frac{(z-\mu)^2}{2\sigma^2}\big] =
\frac{\sigma^2}{2\sigma^2} = \frac12$ (using $\mathbb{E}_q[(z-\mu)^2]=\sigma^2$ by definition of
variance under $q$). The third: $\mathbb{E}_q\big[\frac{z^2}{2}\big] = \frac{\sigma^2+\mu^2}{2}$
(using $\mathbb{E}_q[z^2]=\text{Var}_q(z)+\mathbb{E}_q[z]^2=\sigma^2+\mu^2$). Summing:
$$D_{\text{KL}} = -\frac12\log\sigma^2 - \frac12 + \frac{\sigma^2+\mu^2}{2}
= \frac12\big(\sigma^2+\mu^2-1-\log\sigma^2\big).$$
Writing $\text{logvar}=\log\sigma^2$ (so $\sigma^2=\exp(\text{logvar})$) and negating the sign to
match the code's convention: $-\frac12(1+\text{logvar}-\mu^2-\exp(\text{logvar}))$ — exactly the
notebook's `kl_loss` expression, summed over the (independent, per-dimension) latent
coordinates.

</details>

### 🐍 IMPLEMENT
4. Using `our_nb.means_` and `our_nb.vars_`, compute $P(x_j\mid y{=}0)$ by hand for a single
   test feature value, and confirm it matches
   `norm.pdf(x_j, loc=our_nb.means_[0][j], scale=np.sqrt(our_nb.vars_[0][j]))`.

<details>
<summary>✅ How to know you're right</summary>

Computing the Gaussian density formula
$\frac{1}{\sqrt{2\pi\sigma^2}}\exp\big(-\frac{(x_j-\mu)^2}{2\sigma^2}\big)$ directly with
`our_nb.means_[0][j]` as $\mu$ and `our_nb.vars_[0][j]` as $\sigma^2$ should match
`scipy.stats.norm.pdf(x_j, loc=..., scale=np.sqrt(...))` to full floating-point precision — both
are evaluating the identical univariate Gaussian density at the same point, so any discrepancy
beyond rounding indicates a formula slip (commonly, forgetting to take the square root of the
variance before passing it as `scale`, since `scipy.stats.norm` parameterizes by standard
deviation, not variance).

</details>

### 🤖 APPLY
5. Refit the GMM visualization with 4 components instead of 3 (`n_components=4`) on the same
   3-cluster data. Inspect the resulting means and mixture weights, and compare the fitted
   model's BIC (`gmm.bic(X)`) to the 3-component fit's BIC.

<details>
<summary>✅ What you should observe</summary>

You should see the pattern exercise 2 predicted: two of the four fitted means land close
together (both approximating one true cluster), while `gmm.weights_` still sums to exactly $1$
across all four components. The 4-component model's BIC should come out *higher* (worse) than
the 3-component model's BIC (a representative run: roughly $3192$ vs. $3157$) — BIC's complexity
penalty correctly disfavors the unnecessary extra component even though the 4-component model
technically achieves a (very slightly) higher raw log-likelihood, since BIC explicitly trades
off fit quality against parameter count.

</details>

### 🚀 CHALLENGE
6. In the VAE training loop, set the KL term's weight `beta` to `0` (i.e. optimize only
   `recon_loss`, an autoencoder with no regularization on the latent space at all) and retrain
   from scratch. Sample the same way from `z ~ N(0,I)` afterward and compare the generated
   points to the real data — quantify the comparison (e.g. the mean nearest-neighbor distance
   from each generated point to the closest real data point) rather than only inspecting a
   plot, and also compare the encoder's inferred `mu` values' spread to the `beta=0.25` run's.
   Explain why removing the KL term breaks the *generative* use case even though reconstruction
   still works (arguably even better) — connect this to what the KL term in the ELBO is actually
   enforcing about the encoder's output distribution.

<details>
<summary>🔍 What a strong answer covers</summary>

With `beta=0`, reconstruction loss ends up *lower* than with `beta=0.25` (a representative run:
final `recon_loss` around $0.001$ vs. $0.13$) — removing the KL penalty genuinely helps
reconstruction, exactly as expected, since the encoder is now free to spread `mu(x)` values
across whatever region of latent space is most convenient for the decoder, with no pressure to
stay near the prior $N(0,I)$. But sampling `z` from the prior `N(0,I)` and decoding gives
noticeably worse generated points at `beta=0`: mean nearest-neighbor distance from generated
points to real data comes out roughly $10\times$ larger than at `beta=0.25` (representative run:
around $0.24$ vs. $0.02$), because the encoder's actual `mu` values end up spread far outside
what a standard normal prior would produce (representative run: per-dimension std of `mu(x)`
around $1.9$–$2.2$ at `beta=0`, versus around $0.02$–$0.89$ at `beta=0.25`) — the decoder learned
to associate real data with a region of latent space the prior itself rarely visits, so sampling
from the prior at generation time lands the decoder in an unfamiliar part of its input space. A
strong answer explicitly names this as what the KL term enforces: not "the encoder's *mean*
prediction should be near 0" in isolation, but that the encoder's *entire output distribution*
$q_\phi(z\mid x)$ across the whole dataset should stay close to the prior it will actually be
sampled from at generation time — without that pressure, the encoder and decoder can jointly
reach a compromise that reconstructs training data extremely well while leaving the prior's
support largely unused, which is exactly the failure mode that makes `beta=0` (or KL terms
that vanish via posterior collapse) unusable for actually generating new samples.

</details>

---

## 📚 Further Reading
- Kingma & Welling, ["Auto-Encoding Variational Bayes"](https://arxiv.org/abs/1312.6114) (the original VAE paper)
- Bishop, *Pattern Recognition and Machine Learning*, Ch. 9 (Mixture Models and EM)
- Murphy, *Probabilistic Machine Learning: An Introduction*, Ch. 9 (Naive Bayes)

---

*Next notebook: [Bayesian Deep Learning](11_bayesian_deep_learning.ipynb)*